In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("C:/Users/Owner/Onedrive/Desktop/solar_proj")
DATA = BASE / "data"
MODELS = BASE / "models"


In [ ]:
P_LSTM = np.load(MODELS / "P_LSTM.npy")
P_CNN  = np.load(MODELS / "P_CNN.npy")
y_all  = np.load(MODELS / "modelB_y.npy")

matched = np.load(MODELS / "matched_seq_to_cnn_pairs.npy")


In [ ]:
# Cell 1: Setup + load saved artifacts
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

ROOT = Path(r"C:\Users\Owner\Onedrive\Desktop\solar_proj")
MODELS = ROOT / "models"
DATA = ROOT / "data"


ART_P_LSTM = MODELS / "P_LSTM.npy"
ART_P_CNN  = MODELS / "P_CNN.npy"
ART_Y      = MODELS / "modelB_y.npy"
ART_MATCH  = MODELS / "matched_seq_to_cnn_pairs.npy"
ART_ENSEMBLE = MODELS / "ensemble_logreg.pkl"
SHARP_PQ_LABELED = DATA / "sharp_metadata_labeled.parquet"
SHARP_PQ_FULL = DATA / "sharp_metadata_full.parquet"
CNN_INDEX = DATA / "cnn_arrays" / "cnn_index.npy"

out_dir = MODELS
out_dir.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
# existence checks
for p in [ART_P_LSTM, ART_P_CNN, ART_Y, ART_MATCH, ART_ENSEMBLE, SHARP_PQ_LABELED, SHARP_PQ_FULL, CNN_INDEX]:
    print(p.name, "exists?", p.exists())


p_lstm = np.load(ART_P_LSTM) if ART_P_LSTM.exists() else None
p_cnn  = np.load(ART_P_CNN)  if ART_P_CNN.exists() else None
y_all  = np.load(ART_Y)      if ART_Y.exists() else None
matched_pairs = np.load(ART_MATCH) if ART_MATCH.exists() else None

ensemble_model = None
if ART_ENSEMBLE.exists():
    try:
        ensemble_model = joblib.load(ART_ENSEMBLE)
    except Exception as e:
        print("Could not load ensemble model with joblib:", e)
        ensemble_model = None

# load SHARP metadata (prefer labeled)
sharp_path = SHARP_PQ_LABELED if SHARP_PQ_LABELED.exists() else (SHARP_PQ_FULL if SHARP_PQ_FULL.exists() else None)
if sharp_path is None:
    raise FileNotFoundError("No SHARP parquet found at expected location.")
df_sharp = pd.read_parquet(sharp_path)
df_sharp["T_REC"] = pd.to_datetime(df_sharp["T_REC"], utc=True, errors="coerce")

print(f"Loaded SHARP rows: {len(df_sharp)} range: {df_sharp['T_REC'].min()} -> {df_sharp['T_REC'].max()}")

# quick summaries
if p_lstm is not None: print("P_LSTM shape:", p_lstm.shape, "mean", float(np.nanmean(p_lstm)))
if p_cnn is not None:  print("P_CNN shape:", p_cnn.shape, "mean", float(np.nanmean(p_cnn)))
if matched_pairs is not None: print("matched pairs shape:", matched_pairs.shape)
if ensemble_model is not None: print("Loaded ensemble model type:", type(ensemble_model))


In [ ]:
# Cell 2: Attach P_LSTM, P_CNN, Ensemble probs to SHARP rows
from sklearn.linear_model import LogisticRegression
import numpy as np

# Work on a copy
df = df_sharp.copy()

# Attach P_LSTM: best-effort alignment (you previously used first N)
if p_lstm is not None:
    if len(p_lstm) == len(df):
        df["P_LSTM"] = p_lstm
        print("Attached P_LSTM directly (length match).")
    else:

        n = min(len(p_lstm), len(df))
        df["P_LSTM"] = np.nan
        df.loc[:n-1, "P_LSTM"] = p_lstm[:n]
        print(f"P_LSTM length mismatch: attached first {n} values, remaining NaN.")
else:
    df["P_LSTM"] = np.nan
    print("No P_LSTM available; column filled with NaN.")

# Attach P_CNN via cnn_index and matched index mapping if possible
# Build cnn_index -> (HARPNUM, TIMESTAMP_STR) mapping if file exists
if (CNN_INDEX.exists() or (DATA / "cnn_arrays" / "cnn_index.npy").exists()):
    cnn_index = np.load(CNN_INDEX, allow_pickle=True)
 
    idx_rows = []
    for i, entry in enumerate(cnn_index):
        harp = None; ts = None
        try:
            if isinstance(entry, (list, tuple, np.ndarray)) and len(entry) >= 2:
                harp = int(entry[0])
                ts = entry[1]
            elif isinstance(entry, dict):
                harp = int(entry.get("HARPNUM", entry.get("harp", -1)))
                ts = entry.get("TIMESTAMP_STR", entry.get("T_REC", None))
        except Exception:
            pass
        idx_rows.append({"cnn_idx": int(i), "harp": harp, "raw_ts": ts})
    idx_df = pd.DataFrame(idx_rows)
    # attempt to parse timestamps to UTC where possible (best-effort)
    def _parse_ts(v):
        try:
            return pd.to_datetime(v, utc=True)
        except Exception:
            return pd.NaT
    idx_df["cnn_time"] = idx_df["raw_ts"].apply(_parse_ts)
else:
    idx_df = None
    print("No cnn_index file.")

# Initialize P_CNN column (NaN)
df["P_CNN"] = np.nan

if p_cnn is not None and idx_df is not None:
    # Build mapping of cnn_idx -> prob
    p_cnn_arr = np.asarray(p_cnn)
    idx_df["prob"] = p_cnn_arr

    if matched_pairs is not None:
 
        for seq_idx, cnn_idx in matched_pairs:
            if int(seq_idx) < len(df) and int(cnn_idx) < len(p_cnn_arr):
                df.at[int(seq_idx), "P_CNN"] = float(p_cnn_arr[int(cnn_idx)])
        print("Attached P_CNN to SHARP rows using matched_pairs best-effort mapping.")
    else:
        # fallback: attach P_CNN by cnn_index harp+time matching onto df: nearest time within +/- 15 minutes
        print("No matched_pairs file; attempting best-effort harp+time join (may be slow).")
        if idx_df is not None:
            # build by harp groups
            idx_df = idx_df.sort_values(["harp","cnn_time"])
            for harp, g in idx_df.groupby("harp"):
                # find df rows with same HARPNUM
                mask = df["HARPNUM"] == harp
                if mask.sum() == 0: continue
                times = g["cnn_time"].dropna().values
                if len(times) == 0: continue
                # for each df row for this harp, find nearest cnn_time
                subidx = df[mask].index
                for ridx in subidx:
                    t = df.at[ridx, "T_REC"]
                    if pd.isna(t): continue
                    # compute absolute timedelta minutes
                    diffs = np.abs((g["cnn_time"] - t).astype('timedelta64[m]').astype(float))
                    if len(diffs)==0 or np.all(np.isnan(diffs)): continue
                    argmin = int(np.nanargmin(diffs))
                    if diffs.iloc[argmin] <= 15:
                        prob = float(g.iloc[argmin]["prob"])
                        df.at[ridx, "P_CNN"] = prob
            print("Attached P_CNN via harp+nearest-time join (best-effort).")
else:
    print("P_CNN or cnn_index not available: skipping P_CNN attachment.")

df["P_ENSEMBLE"] = np.nan
if ensemble_model is not None:

    # find rows with both probs
    mask = df["P_LSTM"].notna() & df["P_CNN"].notna()
    if mask.sum() == 0:
        print("No rows have both P_LSTM & P_CNN to apply ensemble.")
    else:
        X_ens = df.loc[mask, ["P_LSTM","P_CNN"]].values
        try:
            probs = ensemble_model.predict_proba(X_ens)[:,1]
            df.loc[mask, "P_ENSEMBLE"] = probs
            print("Computed ensemble probs for", mask.sum(), "rows.")
        except Exception as e:
            print("Ensemble predict_proba failed:", e)
            # try if model is a plain sklearn linear model with coef_ expecting different order
            try:
                # fallback linear combination if coef available
                coef = getattr(ensemble_model, "coef_", None)
                intercept = getattr(ensemble_model, "intercept_", None)
                if coef is not None:
                    # compute logistic(sigmoid(X.dot(coef.T) + intercept))
                    import scipy.special as sc
                    s = X_ens.dot(coef.T) + intercept
                    probs = sc.expit(s.ravel())
                    df.loc[mask, "P_ENSEMBLE"] = probs
                    print("Fallback ensemble logistic applied.")
            except Exception:
                pass
else:
    print("No ensemble model available; P_ENSEMBLE left NaN.")

# Save merged (light) table for Phase2 work
OUT_MERGED = MODELS / "merged_for_phase2.parquet"
cols_to_save = ["HARPNUM","T_REC","P_LSTM","P_CNN","P_ENSEMBLE"]
df[cols_to_save].to_parquet(OUT_MERGED, index=False)
print("Saved merged table to:", OUT_MERGED)


In [ ]:
# Cell 3: Load optional datasets (GOES proton, OMNI/Kp, TEC maps)
RAW = DATA / "raw"
GOES_PROTONS = RAW / "goes_protons.csv"    # optional: time, p10, p30, ...
OMNI_HOURLY  = RAW / "omni_hourly.csv"     # optional: time, kp, dst, b_total...
GIM_DIR      = RAW / "gim"                 # optional netCDF/npys for TEC over region
FLIGHTS_FILE = RAW / "sample_flights.csv"  # optional: lat,lon,alt,time (UTC iso)

goes = pd.read_csv(GOES_PROTONS, parse_dates=['time'], index_col='time') if GOES_PROTONS.exists() else None
omni = pd.read_csv(OMNI_HOURLY, parse_dates=['time'], index_col='time') if OMNI_HOURLY.exists() else None
if goes is not None:
    print("Loaded GOES protons:", goes.shape, "columns:", goes.columns.tolist())
if omni is not None:
    print("Loaded OMNI hourly:", omni.shape, "columns:", omni.columns.tolist())

flights = pd.read_csv(FLIGHTS_FILE, parse_dates=['time']) if FLIGHTS_FILE.exists() else None
if flights is not None:
    print("Loaded flights:", flights.shape)


In [ ]:
# Cell 4: Define risk mapping functions and compute risk columns
import numpy as np

# load merged df
df = pd.read_parquet(MODELS / "merged_for_phase2.parquet")
df["T_REC"] = pd.to_datetime(df["T_REC"], utc=True)

# utility: nearest value lookup for time series with index
def lookup_nearest(series_df, times, default=np.nan):
    # series_df: pandas Series indexed by datetime
    if series_df is None or len(series_df)==0: 
        return np.array([default]*len(times))
    # scipy searchsorted approach
    idxs = series_df.index.searchsorted(times)
    out = []
    for i, t in enumerate(times):
        j = idxs[i]
        # check j and j-1 for nearest
        cand = []
        if j < len(series_df): cand.append((abs((series_df.index[j]-t).total_seconds()), series_df.iloc[j]))
        if j-1 >= 0: cand.append((abs((series_df.index[j-1]-t).total_seconds()), series_df.iloc[j-1]))
        if len(cand)==0: out.append(default)
        else:
            out.append(cand[0][1] if cand[0][0] <= cand[-1][0] else cand[-1][1])
    return np.array(out, dtype=float)

# 1) Radiation risk function
def radiation_risk_from_pfu(p10, lat_deg=60.0, alt_km=350.0, k_rad=1.0):
    # simple proxy: log1p(p10) scaled by polar lat factor and altitude factor
    latf = np.exp(-(np.abs(lat_deg)-60.0)/20.0)  # increases at high lat
    altf = np.exp((alt_km-200.0)/10000.0)
    return k_rad * np.log1p(np.maximum(0.0, p10)) * latf * altf

# 2) GPS risk: prefer TEC (if available) else use geomag index proxy (Kp)
def gps_risk_from_tec(tec_dev, kp=0.0, k_gps=1.0):
    # tec_dev in TECU (if None use kp)
    if np.isnan(tec_dev):
        # fallback: use kp scaled
        return k_gps * (kp/9.0)
    else:
        # normalize tec_dev with a sensible scale (e.g. 5 TECU)
        return k_gps * np.tanh(tec_dev / 5.0)

# 3) Radio (HF) risk: use GOES short X-ray flux if available else use ensemble/LSTM prob
def radio_risk_from_goes(xray, sunlit=1.0, k_radio=1.0):
    # xray in W/m2; use log1p
    return k_radio * sunlit * np.log1p(np.maximum(0.0, xray))

# Attach auxiliary arrays
times = df["T_REC"].values

# Lookup GOES proton p10 if available (nearest)
p10_vals = lookup_nearest(goes['p10'] if goes is not None and 'p10' in goes.columns else None, df["T_REC"]) if goes is not None else np.full(len(df), np.nan)
# Lookup Kp if available
kp_vals = lookup_nearest(omni['Kp'] if omni is not None and 'Kp' in omni.columns else None, df["T_REC"]) if omni is not None else np.full(len(df), np.nan)
# Lookup GOES xray short channel if present
xray_vals = lookup_nearest(goes['xray_short'] if goes is not None and 'xray_short' in goes.columns else None, df["T_REC"]) if goes is not None else np.full(len(df), np.nan)

# Choose best source for "physical trigger" p10:
# Prefer actual GOES p10; else use P_ENSEMBLE if present; else P_LSTM as proxy
phys_p10 = p10_vals.copy()
# if missing, substitute with scaled probabilities (arbitrary mapping: prob*10 pfu)
missing_mask = np.isnan(phys_p10)
if missing_mask.any():
    # use ensemble where available
    ens_sub = df["P_ENSEMBLE"].values
    lstm_sub = df["P_LSTM"].values
    phys_p10[missing_mask] = np.where(~np.isnan(ens_sub[missing_mask]), ens_sub[missing_mask]*10.0,
                                      np.where(~np.isnan(lstm_sub[missing_mask]), lstm_sub[missing_mask]*10.0, 0.0))

lat_default = 60.0
alt_default = 350.0
rad_risk = radiation_risk_from_pfu(phys_p10, lat_deg=lat_default, alt_km=alt_default, k_rad=1.0)


if "TEC_dev" in df.columns:
    tec_dev_arr = df["TEC_dev"].values
else:
    tec_dev_arr = np.full(len(df), np.nan)

gps_risk = np.array([gps_risk_from_tec(td, k) for td, k in zip(tec_dev_arr, np.nan_to_num(kp_vals, nan=0.0))])

# radio risk
radio_risk = np.where(np.isnan(xray_vals),
                      # fallback to ensemble/LSTM as proxy: log1p(prob * scale)
                      np.log1p(np.nan_to_num(df["P_ENSEMBLE"].values, 0.0)*10.0),
                      radio_risk_from_goes(xray_vals, sunlit=1.0, k_radio=1.0))

# normalize each risk to [0,1] roughly via logistic scaling or tanh
def normalize_arr(arr, scale=1.0):
    # robust normalize to 0-1 by tanh of (arr/scale)
    return np.clip(np.tanh(arr / scale), 0.0, 1.0)

df["RADIATION_RISK_raw"] = rad_risk
df["GPS_RISK_raw"] = gps_risk
df["RADIO_RISK_raw"] = radio_risk

# choose scales (these are tunable)
df["RADIATION_RISK"] = normalize_arr(df["RADIATION_RISK_raw"].values, scale=2.0)   # scale chosen empirically
df["GPS_RISK"]       = normalize_arr(df["GPS_RISK_raw"].values, scale=0.5)
df["RADIO_RISK"]     = normalize_arr(df["RADIO_RISK_raw"].values, scale=1.0)

# Final Flight Risk Index R: weighted sum
w1, w2, w3 = 0.4, 0.3, 0.3
df["FLIGHT_RISK"] = w1 * df["RADIATION_RISK"] + w2 * df["GPS_RISK"] + w3 * df["RADIO_RISK"]

print("Computed risk columns. Example head:")
display(df[["T_REC","P_LSTM","P_CNN","P_ENSEMBLE","RADIATION_RISK","GPS_RISK","RADIO_RISK","FLIGHT_RISK"]].head(12))

# Save phase2 merged
OUT_PHASE2 = MODELS / "phase2_merged_with_risk.parquet"
df.to_parquet(OUT_PHASE2, index=False)
print("Saved phase2 merged table:", OUT_PHASE2)


In [ ]:
# Cell 3: Load optional datasets (GOES proton, OMNI/Kp, TEC maps)
RAW = DATA / "raw"
GOES_PROTONS = RAW / "goes_protons.csv"    # optional: time, p10, p30, ...
OMNI_HOURLY  = RAW / "omni_hourly.csv"     # optional: time, kp, dst, b_total...
GIM_DIR      = RAW / "gim"                 # optional netCDF/npys for TEC over region
FLIGHTS_FILE = RAW / "sample_flights.csv"  # optional: lat,lon,alt,time (UTC iso)

goes = pd.read_csv(GOES_PROTONS, parse_dates=['time'], index_col='time') if GOES_PROTONS.exists() else None
omni = pd.read_csv(OMNI_HOURLY, parse_dates=['time'], index_col='time') if OMNI_HOURLY.exists() else None
if goes is not None:
    print("Loaded GOES protons:", goes.shape, "columns:", goes.columns.tolist())
if omni is not None:
    print("Loaded OMNI hourly:", omni.shape, "columns:", omni.columns.tolist())

flights = pd.read_csv(FLIGHTS_FILE, parse_dates=['time']) if FLIGHTS_FILE.exists() else None
if flights is not None:
    print("Loaded flights:", flights.shape)


In [ ]:
# Cell 4: Define risk mapping functions and compute risk columns
import numpy as np

# load merged df
df = pd.read_parquet(MODELS / "merged_for_phase2.parquet")
df["T_REC"] = pd.to_datetime(df["T_REC"], utc=True)

# utility: nearest value lookup for time series with index
def lookup_nearest(series_df, times, default=np.nan):
    # series_df: pandas Series indexed by datetime
    if series_df is None or len(series_df)==0: 
        return np.array([default]*len(times))
    # scipy searchsorted approach
    idxs = series_df.index.searchsorted(times)
    out = []
    for i, t in enumerate(times):
        j = idxs[i]
        # check j and j-1 for nearest
        cand = []
        if j < len(series_df): cand.append((abs((series_df.index[j]-t).total_seconds()), series_df.iloc[j]))
        if j-1 >= 0: cand.append((abs((series_df.index[j-1]-t).total_seconds()), series_df.iloc[j-1]))
        if len(cand)==0: out.append(default)
        else:
            out.append(cand[0][1] if cand[0][0] <= cand[-1][0] else cand[-1][1])
    return np.array(out, dtype=float)

# 1) Radiation risk function
def radiation_risk_from_pfu(p10, lat_deg=60.0, alt_km=350.0, k_rad=1.0):
    # simple proxy: log1p(p10) scaled by polar lat factor and altitude factor
    latf = np.exp(-(np.abs(lat_deg)-60.0)/20.0)  # increases at high lat
    altf = np.exp((alt_km-200.0)/10000.0)
    return k_rad * np.log1p(np.maximum(0.0, p10)) * latf * altf

# 2) GPS risk: prefer TEC (if available) else use geomag index proxy (Kp)
def gps_risk_from_tec(tec_dev, kp=0.0, k_gps=1.0):
    # tec_dev in TECU (if None use kp)
    if np.isnan(tec_dev):
        # fallback: use kp scaled
        return k_gps * (kp/9.0)
    else:
        # normalize tec_dev with a sensible scale (e.g. 5 TECU)
        return k_gps * np.tanh(tec_dev / 5.0)

# 3) Radio (HF) risk: use GOES short X-ray flux if available else use ensemble/LSTM prob
def radio_risk_from_goes(xray, sunlit=1.0, k_radio=1.0):
    # xray in W/m2; use log1p
    return k_radio * sunlit * np.log1p(np.maximum(0.0, xray))

# Attach auxiliary arrays
times = df["T_REC"].values

# Lookup GOES proton p10 if available (nearest)
p10_vals = lookup_nearest(goes['p10'] if goes is not None and 'p10' in goes.columns else None, df["T_REC"]) if goes is not None else np.full(len(df), np.nan)
# Lookup Kp if available
kp_vals = lookup_nearest(omni['Kp'] if omni is not None and 'Kp' in omni.columns else None, df["T_REC"]) if omni is not None else np.full(len(df), np.nan)
# Lookup GOES xray short channel if present
xray_vals = lookup_nearest(goes['xray_short'] if goes is not None and 'xray_short' in goes.columns else None, df["T_REC"]) if goes is not None else np.full(len(df), np.nan)

# Choose best source for "physical trigger" p10:
# Prefer actual GOES p10; else use P_ENSEMBLE if present; else P_LSTM as proxy
phys_p10 = p10_vals.copy()
# if missing, substitute with scaled probabilities (arbitrary mapping: prob*10 pfu)
missing_mask = np.isnan(phys_p10)
if missing_mask.any():
    # use ensemble where available
    ens_sub = df["P_ENSEMBLE"].values
    lstm_sub = df["P_LSTM"].values
    phys_p10[missing_mask] = np.where(~np.isnan(ens_sub[missing_mask]), ens_sub[missing_mask]*10.0,
                                      np.where(~np.isnan(lstm_sub[missing_mask]), lstm_sub[missing_mask]*10.0, 0.0))

lat_default = 60.0
alt_default = 350.0
rad_risk = radiation_risk_from_pfu(phys_p10, lat_deg=lat_default, alt_km=alt_default, k_rad=1.0)

if "TEC_dev" in df.columns:
    tec_dev_arr = df["TEC_dev"].values
else:
    tec_dev_arr = np.full(len(df), np.nan)

gps_risk = np.array([gps_risk_from_tec(td, k) for td, k in zip(tec_dev_arr, np.nan_to_num(kp_vals, nan=0.0))])

# radio risk
radio_risk = np.where(np.isnan(xray_vals),
                      # fallback to ensemble/LSTM as proxy: log1p(prob * scale)
                      np.log1p(np.nan_to_num(df["P_ENSEMBLE"].values, 0.0)*10.0),
                      radio_risk_from_goes(xray_vals, sunlit=1.0, k_radio=1.0))

# normalize each risk to [0,1] roughly via logistic scaling or tanh
def normalize_arr(arr, scale=1.0):
    # robust normalize to 0-1 by tanh of (arr/scale)
    return np.clip(np.tanh(arr / scale), 0.0, 1.0)

df["RADIATION_RISK_raw"] = rad_risk
df["GPS_RISK_raw"] = gps_risk
df["RADIO_RISK_raw"] = radio_risk

# choose scales (these are tunable)
df["RADIATION_RISK"] = normalize_arr(df["RADIATION_RISK_raw"].values, scale=2.0)   # scale chosen empirically
df["GPS_RISK"]       = normalize_arr(df["GPS_RISK_raw"].values, scale=0.5)
df["RADIO_RISK"]     = normalize_arr(df["RADIO_RISK_raw"].values, scale=1.0)

# Final Flight Risk Index R: weighted sum
w1, w2, w3 = 0.4, 0.3, 0.3
df["FLIGHT_RISK"] = w1 * df["RADIATION_RISK"] + w2 * df["GPS_RISK"] + w3 * df["RADIO_RISK"]

print("Computed risk columns. Example head:")
display(df[["T_REC","P_LSTM","P_CNN","P_ENSEMBLE","RADIATION_RISK","GPS_RISK","RADIO_RISK","FLIGHT_RISK"]].head(12))

# Save phase2 merged
OUT_PHASE2 = MODELS / "phase2_merged_with_risk.parquet"
df.to_parquet(OUT_PHASE2, index=False)
print("Saved phase2 merged table:", OUT_PHASE2)


In [ ]:

FLIGHT_FILE = DATA / "raw" / "sample_flights.csv"
if FLIGHT_FILE.exists():
    flights = pd.read_csv(FLIGHT_FILE, parse_dates=['time'])

    df_sharp_indexed = df.set_index("T_REC").sort_index()
    out_rows = []
    for _, row in flights.iterrows():
        t = pd.to_datetime(row['time'], utc=True)
        # find nearest SHARP time
        try:
            nearest_idx = df_sharp_indexed.index.get_indexer([t], method='nearest')[0]
            match_row = df_sharp_indexed.iloc[nearest_idx]
            out_rows.append({
                "flight_time": t,
                "lat": row.get("lat", None),
                "lon": row.get("lon", None),
                "alt_km": row.get("alt_km", None),
                "flight_risk": float(match_row["FLIGHT_RISK"]),
                "radiation_risk": float(match_row["RADIATION_RISK"])
            })
        except Exception:
            out_rows.append({"flight_time": t, "flight_risk": np.nan})
    df_flight_risk = pd.DataFrame(out_rows)
    display(df_flight_risk.head())
    df_flight_risk.to_csv(MODELS / "phase2_flight_track_risks.csv", index=False)
    print("Saved flight track risk mapping.")
else:
    print("No flight track file found; skipping route-level mapping.")


In [ ]:
pip install pandas requests python-dateutil tqdm

In [ ]:
# omni_5min_to_hourly.py
# Download OMNI 5-minute yearly files (omni_5minYYYY.asc) for 2010-2020 from CDAWeb,
# parse them into canonical columns, slice to 2010-05-01..2020-12-31, and
# save raw concatenated CSV + hourly (last-per-hour) CSV + manifest JSON.
#
# Output paths (exact):
# <ROOT>/data/raw/omni/omni_minraw_20100501_20201231.csv
# <ROOT>/data/raw/omni/omni_hourly_20100501_20201231.csv
# <ROOT>/data/raw/omni/omni_manifest_20100501_20201231.json
#
# Requirements: pandas, requests, python-dateutil, tqdm
# pip install pandas requests python-dateutil tqdm

import re
import json
import time
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
from tqdm import tqdm
from dateutil import tz

ROOT = Path(r"C:\Users\Owner\Onedrive\Desktop\solar_proj")
OUT_DIR = (ROOT / "data" / "raw" / "omni")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CDAWEB_BASE = "https://cdaweb.gsfc.nasa.gov/pub/data/omni/high_res_omni/"
YEARS = list(range(2010, 2021))  # inclusive 2010..2020
PROJECT_START = pd.Timestamp("2010-05-01T00:00:00+00:00")
PROJECT_END   = pd.Timestamp("2020-12-31T23:59:00+00:00")

RAW_OUT = OUT_DIR / "omni_minraw_20100501_20201231.csv"
HOURLY_OUT = OUT_DIR / "omni_hourly_20100501_20201231.csv"
MANIFEST_OUT = OUT_DIR / "omni_manifest_20100501_20201231.json"

HEADERS = {"User-Agent": "solar_proj_omni5min_fetch/1.0"}
MAX_RETRIES = 3
DELAY = 0.25


session = requests.Session()
session.headers.update(HEADERS)

def download_yearly_5min(year: int) -> (bool, str, Path):
    """Download omni_5minYYYY.asc into OUT_DIR; return (ok, error_or_url, local_path)."""
    fname = f"omni_5min{year:04d}.asc"
    url = urljoin(CDAWEB_BASE, fname)
    local = OUT_DIR / fname
    if local.exists():
        return True, url, local
    for attempt in range(1, MAX_RETRIES+1):
        try:
            r = session.get(url, stream=True, timeout=120)
            if r.status_code != 200:
                return False, f"HTTP {r.status_code} for {url}", local
            # quick content-type guard
            ctype = r.headers.get("Content-Type", "")
            if "html" in ctype.lower():
                return False, f"Content-Type HTML ({ctype}) for {url}", local
            with open(local, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1<<16):
                    if chunk:
                        fh.write(chunk)
            time.sleep(DELAY)
            return True, url, local
        except Exception as e:
            time.sleep(1 + attempt)
    return False, "download_failed", local

def detect_time_pattern_and_build_ts(df):
    """
    Try to build 'timestamp_utc' from df columns.
    Common OMNI 5-min formats:
      - columns: YYYY MM DD HH MM ...  (5 leading ints)
      - or: YYYY DOY HHMM ...  (Year, DayOfYear, HHMM) - used in some files
    This function attempts both heuristics and returns df with timestamp_utc or None.
    """
    cols = list(df.columns)
    # ensure first few columns are numeric
    # try pattern YYYY MM DD HH MM
    try:
        # if at least 5 cols
        if len(cols) >= 5:
            cand = df.iloc[:, :5].apply(pd.to_numeric, errors='coerce')
            # check plausible year in col0
            if cand.iloc[:,0].between(1900, 2100).any():
                years = cand.iloc[:,0].astype('Int64')
                months = cand.iloc[:,1].astype('Int64')
                days = cand.iloc[:,2].astype('Int64')
                hours = cand.iloc[:,3].astype('Int64')
                mins = cand.iloc[:,4].astype('Int64')
                # try create timestamps rowwise
                ts = []
                for y,mo,d,h,mi in zip(years.fillna(0), months.fillna(0), days.fillna(0), hours.fillna(0), mins.fillna(0)):
                    try:
                        dt = datetime(int(y), int(mo), int(d), int(h), int(mi))
                        ts.append(pd.Timestamp(dt, tz='UTC'))
                    except Exception:
                        ts.append(pd.NaT)
                if pd.Series(ts).notna().sum() > 0:
                    df['timestamp_utc'] = pd.to_datetime(pd.Series(ts))
                    return df
    except Exception:
        pass
    # try YYYY DOY HHMM
    try:
        if len(cols) >= 3:
            cand = df.iloc[:, :3].apply(pd.to_numeric, errors='coerce')
            if cand.iloc[:,0].between(1900, 2100).any():
                years = cand.iloc[:,0].astype('Int64')
                doys = cand.iloc[:,1].astype('Int64')
                hhmm = cand.iloc[:,2].astype('Int64')
                hours = (hhmm // 100).astype('Int64')
                mins = (hhmm % 100).astype('Int64')
                ts = []
                for y,d,h,m in zip(years.fillna(0), doys.fillna(0), hours.fillna(0), mins.fillna(0)):
                    try:
                        dt = pd.to_datetime(f"{int(y)}-{int(d):03d} {int(h):02d}:{int(m):02d}", format="%Y-%j %H:%M")
                        ts.append(pd.Timestamp(dt, tz='UTC'))
                    except Exception:
                        ts.append(pd.NaT)
                if pd.Series(ts).notna().sum() > 0:
                    df['timestamp_utc'] = pd.to_datetime(pd.Series(ts)).dt.tz_localize('UTC')
                    return df
    except Exception:
        pass
    # last resort: look for an ISO-like token column
    for c in cols:
        s = df[c].astype(str)
        if s.str.contains(r'\d{4}-\d{2}-\d{2}').any():
            try:
                df['timestamp_utc'] = pd.to_datetime(df[c], utc=True, errors='coerce')
                if df['timestamp_utc'].notna().sum() > 0:
                    return df
            except Exception:
                pass
    return None

def heuristic_map_columns(df):
    """
    Given a DataFrame with numeric columns (and timestamp_utc), map to:
      V_sw_kms, N_p_cm3, B_total_nT, Bz_gsm_nT, temp_K
    using value-range heuristics and name hints.
    Returns mapped DataFrame with those canonical columns (filled NaN if missing) and mapping dict.
    """
    out = pd.DataFrame()
    out['timestamp_utc'] = df['timestamp_utc']
    numeric_cols = [c for c in df.columns if c != 'timestamp_utc']
    stats = {}
    for c in numeric_cols:
        s = pd.to_numeric(df[c], errors='coerce').dropna()
        if s.empty:
            continue
        stats[c] = {'median': float(s.median()), 'q01': float(s.quantile(0.01)), 'q99': float(s.quantile(0.99))}
    mapped = {'V_sw_kms': None, 'N_p_cm3': None, 'B_total_nT': None, 'Bz_gsm_nT': None, 'temp_K': None}
    # name-based quick mapping
    for c in numeric_cols:
        lc = c.lower()
        if ('speed' in lc or 'v'==lc or 'v_sw' in lc or 'v_x' in lc or 'flow' in lc) and mapped['V_sw_kms'] is None:
            mapped['V_sw_kms'] = c
        if any(k in lc for k in ('dens','n_p','n_p','density','np')) and mapped['N_p_cm3'] is None:
            mapped['N_p_cm3'] = c
        if any(k in lc for k in ('btotal','bt','b_mag','f','abs_b','b_magnitude')) and mapped['B_total_nT'] is None:
            mapped['B_total_nT'] = c
        if 'bz' in lc and mapped['Bz_gsm_nT'] is None:
            mapped['Bz_gsm_nT'] = c
        if 'temp' in lc and mapped['temp_K'] is None:
            mapped['temp_K'] = c
    # value-based mapping
    for c,st in stats.items():
        med = st['median']
        q99 = st['q99']
        lc = c.lower()
        if mapped['V_sw_kms'] is None and 100 <= med <= 2000:
            mapped['V_sw_kms'] = c
        if mapped['N_p_cm3'] is None and 0 <= med <= 200:
            # choose as density candidate
            mapped['N_p_cm3'] = c
        if mapped['B_total_nT'] is None and 0 <= q99 <= 500:
            mapped['B_total_nT'] = c
        if mapped['Bz_gsm_nT'] is None and -1000 <= med <= 1000:
            mapped['Bz_gsm_nT'] = c
        if mapped['temp_K'] is None and 1e2 <= med <= 1e8:
            mapped['temp_K'] = c
    # finalize unique choices (prefer name-based if multiple)
    # create canonical columns
    for canon in mapped:
        col = mapped[canon]
        if col and col in df.columns:
            out[canon] = pd.to_numeric(df[col], errors='coerce')
        else:
            out[canon] = pd.NA
    return out, mapped

frames = []
downloaded = []
errors = []

for year in tqdm(YEARS, desc="years"):
    ok, url_or_err, local = download_yearly_5min(year)
    if not ok:
        errors.append({"year": year, "error": url_or_err})
        continue
    downloaded.append(str(local))
    # read file into pandas: skip comment lines that start with '#'
    try:
        # read all non-comment lines with whitespace delim (no header)
        with open(local, 'r', errors='ignore') as fh:
            lines = fh.readlines()
        # filter out comment lines (start with #) and blank
        data_lines = [ln for ln in lines if ln.strip() and not ln.lstrip().startswith('#')]
        if not data_lines:
            errors.append({"file": str(local), "error": "no_data_lines"})
            continue
        # sample first 2000 lines into a temp string for pandas
        sample = "".join(data_lines[:200000])  # large chunk
        from io import StringIO
        buf = StringIO(sample)
        # try whitespace-delimited read with no header
        df_try = pd.read_csv(buf, delim_whitespace=True, header=None, engine='python')
        # attach all remaining data lines similarly (if file bigger than sample, read full)
        # safer to read full file with same parser
        full_buf = StringIO("".join(data_lines))
        df_full = pd.read_csv(full_buf, delim_whitespace=True, header=None, engine='python')
        df = df_full.copy()
        # assign provisional column names as integers
        df.columns = [f"c{i}" for i in range(df.shape[1])]
        # attempt to build timestamp
        df_ts = detect_time_pattern_and_build_ts(df)
        if df_ts is None:
            errors.append({"file": str(local), "error": "timestamp_detection_failed"})
            continue
        # now map physics columns heuristically
        mapped_df, mapping = heuristic_map_columns(df_ts)
        # filter to project months within the year
        mapped_df['timestamp_utc'] = pd.to_datetime(mapped_df['timestamp_utc'], utc=True, errors='coerce')
        mapped_df = mapped_df.dropna(subset=['timestamp_utc'])
        # keep only rows within PROJECT_START..PROJECT_END
        mapped_df = mapped_df.loc[(mapped_df['timestamp_utc'] >= PROJECT_START) & (mapped_df['timestamp_utc'] <= PROJECT_END)]
        if mapped_df.empty:
            # nothing in this year within project window; skip
            continue
        frames.append(mapped_df)
    except Exception as e:
        errors.append({"file": str(local), "error": str(e)})
    time.sleep(DELAY)

if not frames:
    raise SystemExit("No OMNI 5-min data parsed successfully. Errors: " + json.dumps(errors[:10], indent=2))

# concat, dedupe, sort
df_all = pd.concat(frames, ignore_index=True)
df_all = df_all.drop_duplicates(subset=['timestamp_utc']).sort_values('timestamp_utc').reset_index(drop=True)

# enforce full project window bounds
df_all = df_all.loc[(df_all['timestamp_utc'] >= PROJECT_START) & (df_all['timestamp_utc'] <= PROJECT_END)].copy()

# Save raw per-5min CSV (canonical column names)
# reorder columns
cols_order = ['timestamp_utc','V_sw_kms','N_p_cm3','B_total_nT','Bz_gsm_nT','temp_K']
for c in cols_order:
    if c not in df_all.columns:
        df_all[c] = pd.NA
df_save = df_all[cols_order].copy()
# format timestamp iso
df_save['timestamp_utc'] = df_save['timestamp_utc'].apply(lambda t: t.isoformat() if not pd.isna(t) else pd.NA)

df_save.to_csv(RAW_OUT, index=False)
print("Saved raw CSV:", RAW_OUT)

# Hourly aggregation (last-value-per-hour)
df_hour = df_all.set_index('timestamp_utc').resample('1H').last().reset_index()
# ensure canonical columns
for c in cols_order:
    if c not in df_hour.columns:
        df_hour[c] = pd.NA
# format timestamp iso
df_hour['timestamp_utc'] = df_hour['timestamp_utc'].apply(lambda t: t.isoformat() if not pd.isna(t) else pd.NA)
df_hour = df_hour[cols_order]
df_hour.to_csv(HOURLY_OUT, index=False)
print("Saved hourly CSV:", HOURLY_OUT)

# manifest
manifest = {
    "dataset": "omni_hro_5min_via_cdaweb",
    "path_raw": str(RAW_OUT),
    "path_hourly": str(HOURLY_OUT),
    "first": df_save['timestamp_utc'].iloc[0] if len(df_save)>0 else None,
    "last": df_save['timestamp_utc'].iloc[-1] if len(df_save)>0 else None,
    "rows_raw": int(len(df_save)),
    "rows_hourly": int(len(df_hour)),
    "downloaded": downloaded[:200],
    "errors_sample": errors[:200],
    "notes": "Extracted omni_5minYYYY.asc for years 2010..2020; heuristic mapping used to identify V, N, |B|, Bz (GSM), temp. Hourly aggregation uses last-value-per-hour."
}
with open(MANIFEST_OUT, "w") as fh:
    json.dump(manifest, fh, indent=2)

print("Wrote manifest:", MANIFEST_OUT)
print(json.dumps({k:manifest[k] for k in ('first','last','rows_raw','rows_hourly')}, indent=2))


In [ ]:


import requests
from urllib.parse import urlencode
import pandas as pd
import io
import json
from datetime import datetime
from pathlib import Path
import time
import re
from tqdm import tqdm


ROOT = Path(r"C:\Users\Owner\Onedrive\Desktop\solar_proj")
OUT_DIR = ROOT / "data" / "raw" / "geomag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_START = pd.Timestamp("2010-05-01T00:00:00+00:00")
PROJECT_END   = pd.Timestamp("2020-12-31T23:59:00+00:00")

RAW_OUT = OUT_DIR / "geomag_indices_raw_20100501_to_20201231.csv"
HOURLY_OUT = OUT_DIR / "geomag_indices_hourly_20100501_to_20201231.csv"
MANIFEST_OUT = OUT_DIR / "geomag_indices_manifest_20100501_to_20201231.json"

# OMNIWeb endpoint (nx1.cgi) and form parameters
NX1_URL = "https://omniweb.gsfc.nasa.gov/cgi/nx1.cgi"


VARS = [8, 23]

# Request options
POST_TEMPLATE = {
    "activity": "retrieve",
    "res": "hour",              # hourly resolution
    "spacecraft": "omni2",
    "start_date": "20100501",   # will be overwritten below
    "end_date": "20201231",     # will be overwritten below
    "scale": "Linear",
    "view": 0,
    "table": 0
}
HEADERS = {"User-Agent": "solar_proj_geomag_fetch/1.0 (+https://example)"}
TIMEOUT = 120


session = requests.Session()
session.headers.update(HEADERS)

def format_post_data(start_yyyymmdd: str, end_yyyymmdd: str, vars_list):
    post = POST_TEMPLATE.copy()
    post["start_date"] = start_yyyymmdd
    post["end_date"] = end_yyyymmdd

    data = []
    for k,v in post.items():
        data.append((k,str(v)))
    for vv in vars_list:
        data.append(("vars", str(vv)))
    return data

def fetch_omni_hourly(start, end, vars_list):
    """
    POST to nx1.cgi and return raw text (ASCII table including header).
    start/end are strings YYYYMMDD
    """
    data = format_post_data(start, end, vars_list)
    resp = session.post(NX1_URL, data=data, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.text

def parse_omni_ascii_table(text, vars_count):
    """
    Parse OMNIWeb ASCII response into DataFrame.
    Heuristics:
      - Skip comment lines beginning with '#'
      - Data lines usually start with year (YYYY) or YYYYMMDD or year/doY
      - For hourly resolution, common format: YYYY MM DD hh val1 val2 ...
        or YYYY DOY HHMM val1 val2 ...
    Returns dataframe with columns: timestamp_utc, col1, col2, ... (col order = vars requested)
    """
    lines = text.splitlines()
    data_lines = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if s.startswith('#'):
            continue
        # sometimes OMNI pages include a footer or format description after data; break if footer begins
        # detect lines that are purely non-numeric (e.g., "END" or "Format:")
        if not re.search(r'\d', s):
            continue
        # filter lines that look like data: begin with digit (year) or whitespace+digit
        if re.match(r'^\s*\d', s):
            data_lines.append(s)
    if not data_lines:
        raise ValueError("No candidate data lines parsed from response.")
 
    sample_tokens = re.split(r'\s+', data_lines[0].strip())
    # If token count >= (4 + vars_count), assume yyyy mm dd hh ...
    df_rows = []
    for ln in data_lines:
        toks = re.split(r'\s+', ln.strip())
        # skip lines that are too short
        if len(toks) < 3 + vars_count:
            continue
        df_rows.append(toks)
    if not df_rows:
        raise ValueError("Data lines present but none matched expected column count.")
    # Try two parsing styles:
    # Style A: YYYY MM DD HH <vars...>
    # Style B: YYYY DOY HHMM <vars...>
    # Style C: YYYYMMDD or other compact forms -> try to detect and parse
    def try_style_A(row):
        # expects at least 4 + vars_count tokens
        if len(row) < 4 + vars_count:
            return None
        try:
            y = int(row[0]); mm = int(row[1]); dd = int(row[2]); hh = int(row[3])
            ts = pd.Timestamp(datetime(y, mm, dd, hh), tz='UTC')
            values = []
            for i in range(vars_count):
                idx = 4 + i
                try:
                    val = float(row[idx])
                except:
                    val = pd.NA
                values.append(val)
            return (ts, values)
        except Exception:
            return None
    def try_style_B(row):
        # expects at least 3+vars tokens; row[1] = doy; row[2] = hhmm
        if len(row) < 3 + vars_count:
            return None
        try:
            y = int(row[0]); doy = int(row[1]); hhmm = int(row[2])
            hh = hhmm // 100
            mm = hhmm % 100
            # build timestamp from year + day-of-year
            dt = pd.to_datetime(f"{y}-{doy:03d} {hh:02d}:{mm:02d}", format="%Y-%j %H:%M")
            ts = pd.Timestamp(dt, tz='UTC')
            values = []
            for i in range(vars_count):
                idx = 3 + i
                try:
                    val = float(row[idx])
                except:
                    val = pd.NA
                values.append(val)
            return (ts, values)
        except Exception:
            return None
    # iterate rows and attempt both styles; collect whichever style yields >50% success
    parsed = []
    style_counts = {"A":0,"B":0}
    for row in df_rows:
        rA = try_style_A(row)
        if rA is not None:
            parsed.append(rA)
            style_counts["A"] += 1
            continue
        rB = try_style_B(row)
        if rB is not None:
            parsed.append(rB)
            style_counts["B"] += 1
            continue
    # If none parsed, try alternate detection for compact date (YYYYMMDDHHMM)
    if not parsed:
        parsed2 = []
        for row in df_rows:
            # find first token that contains 8+ digits
            try:
                tok0 = row[0]
                m = re.match(r'^(20\d{6})(\d{2})?$', tok0)
                if m:
                    ymd = m.group(1)
                    rest = tok0[len(ymd):]
                    # if rest has hour/min use it; else try next token
                    # fallback: skip
                    continue
            except:
                pass
        if not parsed2:
            raise ValueError("Failed to parse rows with known time formats.")
        parsed = parsed2

    # build DataFrame
    timestamps = [p[0] for p in parsed]
    values = [p[1] for p in parsed]
    vals_df = pd.DataFrame(values, columns=[f"var{i+1}" for i in range(vars_count)])
    df = pd.DataFrame({"timestamp_utc":timestamps})
    df = pd.concat([df.reset_index(drop=True), vals_df.reset_index(drop=True)], axis=1)
    # set index and sort/dedup
    df = df.drop_duplicates(subset=['timestamp_utc']).sort_values('timestamp_utc').reset_index(drop=True)
    return df


print("Posting request to OMNIWeb for hourly Kp & Dst...")
start = PROJECT_START.strftime("%Y%m%d")
end = PROJECT_END.strftime("%Y%m%d")
text = fetch_omni_hourly(start, end, VARS)

# Parse returned ASCII
df = parse_omni_ascii_table(text, vars_count=len(VARS))

# naming: var1 -> Kp*10, var2 -> Dst
# convert Kp*10 to canonical kp (divide by 10)
# Provide columns: timestamp_utc (ISO8601 UTC), kp, dst_nT
df = df.rename(columns={"var1":"kp_x10","var2":"dst_nT"})
# safe numeric coercion
df['kp_x10'] = pd.to_numeric(df['kp_x10'], errors='coerce')
df['dst_nT'] = pd.to_numeric(df['dst_nT'], errors='coerce')
df['kp'] = df['kp_x10'] / 10.0
# canonical timestamp formatting
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True, errors='coerce')
df = df.loc[(df['timestamp_utc'] >= PROJECT_START) & (df['timestamp_utc'] <= PROJECT_END)].copy()
# Keep canonical columns and formatting
out_raw = df[['timestamp_utc','kp','dst_nT']].copy()
out_raw['timestamp_utc'] = out_raw['timestamp_utc'].apply(lambda t: t.isoformat())

# Save raw file (hourly, as returned)
raw_path = RAW_OUT
out_raw.to_csv(raw_path, index=False)
print("Saved raw geomag file:", raw_path)

# Now create canonical hourly index and ensure full hourly coverage
hour_index = pd.date_range(start=PROJECT_START, end=PROJECT_END, freq='1H', tz='UTC')
df_hour = df.set_index('timestamp_utc').reindex(hour_index).sort_index()
# df_hour currently has kp and dst_nT columns; fill missing with NaN
df_hour = df_hour[['kp','dst_nT']].reset_index().rename(columns={'index':'timestamp_utc'})

# apply last-value-per-hour if duplicates exist: originally data is hourly so this is just a safety step
df_hour['timestamp_utc'] = pd.to_datetime(df_hour['timestamp_utc'], utc=True)
# format timestamp ISO8601
df_hour['timestamp_utc'] = df_hour['timestamp_utc'].apply(lambda t: t.isoformat())
hourly_path = HOURLY_OUT
df_hour.to_csv(hourly_path, index=False)
print("Saved hourly (canonical) geomag file:", hourly_path)

# manifest
manifest = {
    "dataset":"geomag_indices_kp_dst_omniweb",
    "path_raw": str(raw_path),
    "path_hourly": str(hourly_path),
    "first": out_raw['timestamp_utc'].iloc[0] if len(out_raw)>0 else None,
    "last": out_raw['timestamp_utc'].iloc[-1] if len(out_raw)>0 else None,
    "rows_raw": int(len(out_raw)),
    "rows_hourly": int(len(df_hour)),
    "vars_requested": VARS,
    "notes": "Kp requested as vars=8 (Kp*10) and Dst as vars=23 per OMNIWeb example. Kp converted to canonical scale by dividing by 10. If different mapping is desired change VARS list."
}
with open(MANIFEST_OUT, "w") as fh:
    json.dump(manifest, fh, indent=2)

print("Wrote manifest:", MANIFEST_OUT)
print(json.dumps({k:manifest[k] for k in ('first','last','rows_raw','rows_hourly')}, indent=2))


In [ ]:
import requests
from pathlib import Path
from datetime import datetime, timedelta
import logging


ROOT = Path(r"C:\Users\Owner\Onedrive\Desktop\solar_proj")
OUT_DIR = ROOT / "data" / "raw" / "tec" / "gim_ionex"
OUT_DIR.mkdir(parents=True, exist_ok=True)

START = datetime(2010, 5, 1)
END   = datetime(2020, 12, 31)

BASE = "https://cddis.nasa.gov/archive/gnss/products/ionex"

LOG_FILE = OUT_DIR / "download_log.txt"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s %(message)s"
)

session = requests.Session()



def doy(dt):
    return dt.timetuple().tm_yday

def build_filename(dt):
    yy = str(dt.year)[2:]
    ddd = f"{doy(dt):03d}"
    return f"igsg{ddd}0.{yy}i.Z"

def build_url(dt):
    year = dt.year
    ddd = f"{doy(dt):03d}"
    fname = build_filename(dt)
    return f"{BASE}/{year}/{ddd}/{fname}", fname



cur = START
total = 0
ok = 0
missing = 0

while cur <= END:

    url, fname = build_url(cur)

    year_dir = OUT_DIR / str(cur.year)
    year_dir.mkdir(parents=True, exist_ok=True)

    out_file = year_dir / fname

    if out_file.exists():
        logging.info(f"SKIP EXISTS {fname}")
        ok += 1
        cur += timedelta(days=1)
        continue

    try:
        r = session.get(url, stream=True, timeout=60)


        ctype = r.headers.get("Content-Type", "")

        if "text/html" in ctype.lower():
            logging.warning(f"LOGIN PAGE NOT FILE {fname}")
            missing += 1
            cur += timedelta(days=1)
            continue

        if r.status_code != 200:
            logging.warning(f"HTTP {r.status_code} {fname}")
            missing += 1
            cur += timedelta(days=1)
            continue

        with open(out_file, "wb") as f:
            for chunk in r.iter_content(8192):
                if chunk:
                    f.write(chunk)

        logging.info(f"DOWNLOADED {fname}")
        ok += 1

    except Exception as e:
        logging.error(f"ERROR {fname} {e}")
        missing += 1

    total += 1
    cur += timedelta(days=1)

print("DONE")
print("Downloaded:", ok)
print("Missing:", missing)
print("Total processed:", total)


In [ ]:


from sunpy.net import Fido, attrs as a
import sunpy.timeseries as ts
import pandas as pd
from pathlib import Path



ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")

RAW_DIR = ROOT / "data" / "raw" / "goes_xray"
OUT_FILE = ROOT / "data" / "processed" / "goes_xray" / \
           "goes_xray_1min_20100501_20201231.parquet"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

START_YEAR = 2010
END_YEAR   = 2020

all_dfs = []



for year in range(START_YEAR, END_YEAR + 1):

    start = f"{year}-01-01"
    end   = f"{year}-12-31"

    # enforce your exact Phase-2 window
    if year == 2010:
        start = "2010-05-01"
    if year == 2020:
        end = "2020-12-31"

    print(f"\n🔎 Searching GOES XRS for {start} → {end}")

    result = Fido.search(
        a.Time(start, end),
        a.Instrument("XRS")
    )

    if len(result) == 0:
        print("⚠️ No data returned — skipping year")
        continue

    print("⬇️ Fetching files...")
    files = Fido.fetch(result, path=str(RAW_DIR))

    print("📊 Building TimeSeries...")
    goes = ts.TimeSeries(files, concatenate=True)

    df = goes.to_dataframe()


    df = df.rename(columns={
        "xrsa": "xrs_short_Wm2",   # 0.05–0.4 nm
        "xrsb": "xrs_long_Wm2"     # 0.1–0.8 nm
    })

    df.index.name = "timestamp_utc"
    df = df.reset_index()

    all_dfs.append(df)


print("\n🔗 Merging yearly datasets...")

df_all = pd.concat(all_dfs, ignore_index=True)

# enforce exact window
df_all = df_all[
    (df_all["timestamp_utc"] >= "2010-05-01") &
    (df_all["timestamp_utc"] <= "2020-12-31")
]

# keep only required columns
df_all = df_all[
    ["timestamp_utc", "xrs_long_Wm2", "xrs_short_Wm2"]
]


print("💾 Saving parquet...")

df_all.to_parquet(OUT_FILE, index=False)

print("\n✅ DONE — GOES XRS dataset created:")
print(OUT_FILE)
print("Rows:", len(df_all))


In [ ]:
import os
print(os.getcwd())


In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Owner\Onedrive\Desktop\solar_proj")
print((ROOT / "data" / "raw" / "goes_xray").exists())
print(list((ROOT / "data" / "raw" / "goes_xray").glob("*"))[:5])


In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Owner\Onedrive\Desktop\solar_proj")

files = list(ROOT.rglob("sci_xrsf*.nc"))
print("Found:", len(files))
print(files[:5])


In [ ]:
import requests
import re
from pathlib import Path



ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")
RAW_DIR  = ROOT / "data" / "raw" / "goes_protons"

RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/avg/"

session = requests.Session()
session.headers.update({"User-Agent":"solar_proj_goes_eps"})


def list_dirs(url):
    """Return list of subdirectory names under a NOAA directory URL."""
    r = session.get(url)
    if r.status_code != 200:
        return []
    return re.findall(r'href="([^"]+/)"', r.text)

def list_files(url):
    """Return list of .nc files under a NOAA directory URL."""
    r = session.get(url)
    if r.status_code != 200:
        return []
    return re.findall(r'href="([^"]+\.nc)"', r.text)


print("Scanning NOAA avg directory for year folders...")

year_dirs = []
for entry in list_dirs(BASE):
    if re.fullmatch(r"\d{4}/", entry):
        year_dirs.append(entry)

print("Year folders found:", year_dirs)


eps_urls = []

for year in year_dirs:
    year_url = BASE + year

    for month in list_dirs(year_url):
        # Only valid months "01/", "02/", ..., "12/"
        if not re.fullmatch(r"(0[1-9]|1[0-2])/", month):
            continue

        month_url = year_url + month

        for sat in list_dirs(month_url):
            # Only satellite folders (e.g., "goes10/", "goes11/", etc.)
            if not sat.startswith("goes"):
                continue

            # Navigate to the netcdf subfolder
            netcdf_url = month_url + sat + "netcdf/"

            # Collect all EPS 5-minute netcdf files
            for f in list_files(netcdf_url):
                if "_eps_5m_" in f:
                    eps_urls.append(netcdf_url + f)

print("Total potential EPS files discovered:", len(eps_urls))


downloaded = 0
skipped = 0

for url in eps_urls:
    fname = url.split("/")[-1]
    out = RAW_DIR / fname

    # If the file already exists, skip it
    if out.exists():
        skipped += 1
        continue

    # Attempt to download
    r = session.get(url, stream=True)
    if r.status_code == 200:
        with open(out, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        print("✔ Saved:", fname)
        downloaded += 1

print(f"\nDownload complete — {downloaded} new files saved, {skipped} skipped.")
print("RAW proton files in directory:", RAW_DIR)


In [ ]:
import pandas as pd
from pathlib import Path



FLARE_IN  = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj\models\merged_for_phase2.parquet")
FLARE_OUT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj\data\processed\phase1_flare_predictions.parquet")

df = pd.read_parquet(FLARE_IN)

# Rename timestamp column
df = df.rename(columns={"T_REC": "timestamp_utc"})

# Use ensemble probability if available
df["flare_probability"] = df["P_ENSEMBLE"]

# Ensure datetime and UTC
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

# Resample to hourly cadence
df_hour = (
    df.set_index("timestamp_utc")
      .resample("1H")
      .mean()
      .reset_index()
)

# Save
FLARE_OUT.parent.mkdir(parents=True, exist_ok=True)
df_hour.to_parquet(FLARE_OUT, index=False)

print("Saved hourly phase1 flare predictions:", FLARE_OUT)
print("Rows:", len(df_hour))
print("Start:", df_hour["timestamp_utc"].min())
print("End:", df_hour["timestamp_utc"].max())


In [ ]:
import xarray as xr
import pandas as pd
from pathlib import Path
from tqdm import tqdm



ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")

XRS_RAW_DIR  = ROOT / "data" / "raw" / "goes_xray"
XRS_PROC_DIR = ROOT / "data" / "processed" / "goes_xray"
XRS_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Output files
XRS_1MIN_OUT   = XRS_PROC_DIR / "goes_xray_1min_20100501_20201231.parquet"
XRS_HOURLY_OUT = XRS_PROC_DIR / "goes_xray_hourly_20100501_20201231.parquet"


all_dfs = []

files = sorted(XRS_RAW_DIR.glob("*.nc"))
print(f"Found {len(files)} GOES XRS NetCDF files")

for f in tqdm(files, desc="Loading GOES XRS"):

    try:
        ds = xr.open_dataset(f)

        # Extract only the irradiance flux variables
        df = ds[["a_flux", "b_flux"]].to_dataframe().reset_index()

        if df.empty:
            continue

        # Rename to standardized names
        df = df.rename(columns={
            "time": "timestamp_utc",
            "a_flux": "xrs_short_Wm2",
            "b_flux": "xrs_long_Wm2"
        })

        df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

        # Keep only our project window
        mask = (
            (df["timestamp_utc"] >= "2010-05-01") &
            (df["timestamp_utc"] <= "2020-12-31 23:59:59")
        )
        df = df.loc[mask]

        if not df.empty:
            all_dfs.append(df)

    except Exception as e:
        print("⛔ Skip:", f.name, e)



if len(all_dfs) == 0:
    raise RuntimeError("No XRS data extracted — check folder path or NetCDF contents.")

df_all = pd.concat(all_dfs, ignore_index=True)

print("Total 1-minute XRS rows (after window trim):", len(df_all))



df_all.to_parquet(XRS_1MIN_OUT, index=False)
print("✅ Saved 1-min GOES XRS parquet:", XRS_1MIN_OUT)



df_hourly = (
    df_all
    .set_index("timestamp_utc")
    .resample("1H")
    .max()
    .reset_index()
)

df_hourly.to_parquet(XRS_HOURLY_OUT, index=False)
print("✅ Saved hourly GOES XRS parquet:", XRS_HOURLY_OUT)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")

START = "2010-05-01"
END   = "2020-12-31"

FLARE_FILE = ROOT / "data" / "processed" / "phase1_flare_predictions.parquet"

XRS_FILE = ROOT / "data" / "processed" / "goes_xray" / \
           "goes_xray_hourly_20100501_20201231.parquet"

OMNI_FILE = ROOT / "data" / "raw" / "omni" / \
            "omni_hourly_20100501_20201231.csv"

GEOMAG_FILE = ROOT / "data" / "raw" / "geomag" / \
              "geomag_indices_hourly_20100501_to_20201231.csv"

TEC_FILE = ROOT / "data" / "processed" / "tec" / \
           "tec_hourly_20100501_to_20201231.csv"

OUT_FILE = ROOT / "data" / "processed" / \
           "phase2_master_20100501_20201231.parquet"



flare = pd.read_parquet(FLARE_FILE)
flare["timestamp_utc"] = pd.to_datetime(flare["timestamp_utc"], utc=True)

flare = (
    flare
    .set_index("timestamp_utc")
    .resample("1h")
    .mean()
    .reset_index()
)

flare = flare[["timestamp_utc", "flare_probability"]]



xrs = pd.read_parquet(XRS_FILE)
xrs["timestamp_utc"] = pd.to_datetime(xrs["timestamp_utc"], utc=True)

xrs["log_xrs_long"] = np.log10(
    xrs["xrs_long_Wm2"].clip(lower=1e-10)
)



omni = pd.read_csv(OMNI_FILE)
omni["timestamp_utc"] = pd.to_datetime(omni["timestamp_utc"], utc=True)

omni = omni[[
    "timestamp_utc",
    "Bz_gsm_nT",
    "V_sw_kms",
    "N_p_cm3"
]].rename(columns={
    "Bz_gsm_nT": "Bz",
    "V_sw_kms": "Vsw",
    "N_p_cm3": "Density"
})



geomag = pd.read_csv(GEOMAG_FILE)
geomag["timestamp_utc"] = pd.to_datetime(geomag["timestamp_utc"], utc=True)

geomag = geomag[[
    "timestamp_utc",
    "kp",
    "dst_nT"
]].rename(columns={
    "kp": "Kp",
    "dst_nT": "Dst"
})


tec = pd.read_csv(TEC_FILE)

# The timestamp is in 'Unnamed: 0'
tec = tec.rename(columns={"Unnamed: 0": "timestamp_utc"})
tec["timestamp_utc"] = pd.to_datetime(tec["timestamp_utc"], utc=True)

tec = tec[[
    "timestamp_utc",
    "vtec_global_mean"
]].rename(columns={
    "vtec_global_mean": "tec_global_mean"
})


def trim(df):
    return df[
        (df["timestamp_utc"] >= START) &
        (df["timestamp_utc"] <= END)
    ]

flare  = trim(flare)
xrs    = trim(xrs)
omni   = trim(omni)
geomag = trim(geomag)
tec    = trim(tec)



df = flare.merge(xrs, on="timestamp_utc", how="inner")
df = df.merge(omni, on="timestamp_utc", how="inner")
df = df.merge(geomag, on="timestamp_utc", how="inner")
df = df.merge(tec, on="timestamp_utc", how="inner")

df = df.sort_values("timestamp_utc").reset_index(drop=True)


df.to_parquet(OUT_FILE, index=False)

print("\n✅ PHASE-2 MASTER DATASET CREATED")
print("Saved:", OUT_FILE)
print("Rows:", len(df))
print("Start:", df["timestamp_utc"].min())
print("End:", df["timestamp_utc"].max())
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")

MASTER_FILE = ROOT / "data" / "processed" / \
              "phase2_master_20100501_20201231.parquet"

df = pd.read_parquet(MASTER_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

print("Loaded master dataset")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())


In [ ]:


# Use log scale already computed
df["hf_blackout_risk"] = (
    (df["log_xrs_long"] + 8) / 4
).clip(0, 1)



# Bz negative component (southward IMF)
df["bz_negative"] = np.clip(-df["Bz"], 0, None)

# 3-hour sustained southward Bz
df["bz_sustained"] = (
    df["bz_negative"]
    .rolling(3, min_periods=1)
    .mean()
)

# TEC anomaly (deviation from 7-day mean)
df["tec_7d_mean"] = (
    df["tec_global_mean"]
    .rolling(24*7, min_periods=24)
    .mean()
)

df["tec_anomaly"] = df["tec_global_mean"] - df["tec_7d_mean"]

# GPS disturbance index (scaled composite)
df["gps_disturbance_index"] = (
    0.4 * (df["Kp"] / 9) +
    0.3 * (np.abs(df["Dst"]) / 200) +
    0.2 * (df["bz_sustained"] / 10) +
    0.1 * (df["tec_anomaly"] / 20)
).clip(0, 1)


df["radiation_proxy"] = (
    df["flare_probability"] *
    df["bz_negative"] *
    df["Vsw"]
)

# normalize radiation proxy
rad_max = df["radiation_proxy"].quantile(0.99)
df["radiation_proxy"] = (df["radiation_proxy"] / rad_max).clip(0,1)


In [ ]:
import numpy as np
import pandas as pd


for col in [
    "hf_blackout_risk",
    "gps_disturbance_index",
    "radiation_proxy"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)


rad_max = df["radiation_proxy"].max()

if rad_max == 0 or pd.isna(rad_max):
    df["radiation_proxy"] = 0
else:
    df["radiation_proxy"] = df["radiation_proxy"] / rad_max


df["risk_score"] = (
    0.4 * df["hf_blackout_risk"] +
    0.4 * df["gps_disturbance_index"] +
    0.2 * df["radiation_proxy"]
)

df["risk_score"] = df["risk_score"].clip(0, 1)

# Final safety: eliminate any remaining NaN/inf
df["risk_score"] = df["risk_score"].replace([np.inf, -np.inf], 0).fillna(0)


def classify_risk(score):
    if score < 0.3:
        return "Low"
    elif score < 0.6:
        return "Moderate"
    else:
        return "High"

df["risk_category"] = df["risk_score"].apply(classify_risk)



print("\nNaNs in risk_score:", df["risk_score"].isna().sum())
print("\nRisk category distribution:")
print(df["risk_category"].value_counts())

print("\nSample rows:")
print(df[["timestamp_utc","risk_score","risk_category"]].head())


In [ ]:
drivers = [
    "hf_blackout_risk",
    "gps_disturbance_index",
    "radiation_proxy"
]

df["dominant_driver"] = df[drivers].idxmax(axis=1)

print("Driver counts:")
print(df["dominant_driver"].value_counts())


In [ ]:
OUT_RISK = ROOT / "data" / "processed" / \
           "phase2_aviation_risk_20100501_20191231.parquet"

df.to_parquet(OUT_RISK, index=False)

print("Saved aviation risk dataset:")
print(OUT_RISK)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.plot(df["timestamp_utc"], df["risk_score"])
plt.title("Aviation Risk Score (2010–2019)")
plt.ylabel("Risk Score (0–1)")
plt.xlabel("Time")
plt.show()


In [ ]:
storm = df[
    (df["timestamp_utc"] >= "2017-09-01") &
    (df["timestamp_utc"] <= "2017-09-10")
]

plt.figure(figsize=(12,4))
plt.plot(storm["timestamp_utc"], storm["risk_score"])
plt.title("Risk During 2017-09 Solar Storm")
plt.show()


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

OUT_DIR = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj\models\phase2_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(14,5))
plt.plot(df["timestamp_utc"], df["risk_score"])
plt.title("Aviation Risk Score (2010–2019)")
plt.ylabel("Risk Score (0–1)")
plt.xlabel("Time")
plt.tight_layout()

plt.savefig(OUT_DIR / "risk_timeseries_2010_2019.png", dpi=300)
plt.close()


In [ ]:
storm = df[
    (df["timestamp_utc"] >= "2017-09-01") &
    (df["timestamp_utc"] <= "2017-09-10")
]

plt.figure(figsize=(12,4))
plt.plot(storm["timestamp_utc"], storm["risk_score"])
plt.title("Risk During 2017-09 Solar Storm")
plt.ylabel("Risk Score")
plt.xlabel("Time")
plt.tight_layout()

plt.savefig(OUT_DIR / "risk_2017_sept_storm.png", dpi=300)
plt.close()

print("Plots saved to:", OUT_DIR)


In [ ]:
import numpy as np

df_flight = df.copy()

def apply_latitude_factor(score, lat_type):
    if lat_type == "polar":
        return score * 1.4
    elif lat_type == "mid":
        return score * 1.0
    elif lat_type == "low":
        return score * 0.7
    else:
        return score

df_flight["risk_polar"] = df_flight["risk_score"].apply(lambda x: apply_latitude_factor(x, "polar"))
df_flight["risk_mid"]   = df_flight["risk_score"].apply(lambda x: apply_latitude_factor(x, "mid"))
df_flight["risk_low"]   = df_flight["risk_score"].apply(lambda x: apply_latitude_factor(x, "low"))

# Cap at 1.0
df_flight[["risk_polar","risk_mid","risk_low"]] = \
    df_flight[["risk_polar","risk_mid","risk_low"]].clip(0,1)

print("Flight risk profiles created.")


In [ ]:
def classify(score):
    if score < 0.3:
        return "Low"
    elif score < 0.6:
        return "Moderate"
    else:
        return "High"

df_flight["cat_polar"] = df_flight["risk_polar"].apply(classify)
df_flight["cat_mid"]   = df_flight["risk_mid"].apply(classify)
df_flight["cat_low"]   = df_flight["risk_low"].apply(classify)

print("Polar risk distribution:")
print(df_flight["cat_polar"].value_counts())


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj\models\phase2_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(14,6))
plt.plot(df_flight["timestamp_utc"], df_flight["risk_polar"], label="Polar")
plt.plot(df_flight["timestamp_utc"], df_flight["risk_mid"], label="Mid-latitude")
plt.plot(df_flight["timestamp_utc"], df_flight["risk_low"], label="Low-latitude")

plt.legend()
plt.title("Aviation Risk by Flight Latitude (2010–2019)")
plt.ylabel("Risk Score")
plt.xlabel("Time")
plt.tight_layout()

plt.savefig(OUT_DIR / "flight_latitude_risk_comparison.png", dpi=300)
plt.close()

print("Flight comparison plot saved.")


In [ ]:
storm = df_flight[
    (df_flight["timestamp_utc"] >= "2017-09-05") &
    (df_flight["timestamp_utc"] <= "2017-09-08")
]

plt.figure(figsize=(12,5))
plt.plot(storm["timestamp_utc"], storm["risk_polar"], label="Polar Flight")
plt.plot(storm["timestamp_utc"], storm["risk_mid"], label="Mid-Latitude Flight")

plt.legend()
plt.title("Flight Risk During Sept 2017 Solar Storm")
plt.ylabel("Risk Score")
plt.xlabel("Time")
plt.tight_layout()

plt.savefig(OUT_DIR / "storm_flight_simulation_2017.png", dpi=300)
plt.close()

print("Storm simulation saved.")


In [ ]:
def operational_action(score):
    if score < 0.3:
        return "Normal Ops"
    elif score < 0.6:
        return "Monitor / Fuel Adjust"
    else:
        return "Consider Reroute"

df_flight["polar_decision"] = df_flight["risk_polar"].apply(operational_action)


In [ ]:
print("Sept 2017 Storm Summary:")
print(storm[["risk_polar", "risk_mid", "risk_low"]].describe())


In [ ]:
peak = df_flight.loc[df_flight["risk_polar"].idxmax()]
print("\nPeak Polar Risk Event:")
print(peak[["timestamp_utc", "risk_polar"]])


In [ ]:


import numpy as np

df_components = df.copy()

# HF blackout proxy (based on XRS log flux)
df_components["hf_blackout"] = (
    df_components["log_xrs_long"] - df_components["log_xrs_long"].min()
) / (
    df_components["log_xrs_long"].max() - df_components["log_xrs_long"].min()
)

# GPS disturbance proxy
df_components["gps_disturbance"] = (
    0.4 * df_components["Kp"] +
    0.3 * abs(df_components["Dst"]) / df_components["Dst"].abs().max() +
    0.3 * abs(df_components["Bz"]) / df_components["Bz"].abs().max()
)

# Normalize GPS disturbance
df_components["gps_disturbance"] = (
    df_components["gps_disturbance"] -
    df_components["gps_disturbance"].min()
) / (
    df_components["gps_disturbance"].max() -
    df_components["gps_disturbance"].min()
)

# Radiation proxy
df_components["radiation_proxy"] = (
    df_components["flare_probability"] *
    np.maximum(0, -df_components["Bz"]) *
    df_components["Vsw"]
)

# Normalize radiation
df_components["radiation_proxy"] = (
    df_components["radiation_proxy"] -
    df_components["radiation_proxy"].min()
) / (
    df_components["radiation_proxy"].max() -
    df_components["radiation_proxy"].min()
)

print("Drivers rebuilt.")


In [ ]:
df_components["dominant_driver"] = df_components[
    ["hf_blackout", "gps_disturbance", "radiation_proxy"]
].idxmax(axis=1)

print(df_components["dominant_driver"].value_counts())


In [ ]:
import numpy as np
import pandas as pd

# df must have: timestamp_utc, risk_score (0..1), risk_category
df = df.copy()

def altitude_factor_ft(alt_ft, mid=36000, steep=3500, min_fac=0.60, max_fac=1.35):
    """
    Smooth altitude scaling factor.
    - around 36k ft factor ~ midpoint
    - saturates toward max_fac by ~45k
    - saturates toward min_fac by ~25-30k
    """
    alt_ft = np.asarray(alt_ft, dtype=float)
    z = (alt_ft - mid) / steep
    sig = 1 / (1 + np.exp(-z))
    return min_fac + (max_fac - min_fac) * sig

# Choose representative flight levels
ALT_POLAR_FT = 41000   # long-haul polar often higher
ALT_MID_FT   = 37000
ALT_LOW_FT   = 33000

df["alt_factor_polar"] = altitude_factor_ft(ALT_POLAR_FT)
df["alt_factor_mid"]   = altitude_factor_ft(ALT_MID_FT)
df["alt_factor_low"]   = altitude_factor_ft(ALT_LOW_FT)

# Altitude-adjusted risk (clip to 0..1)
df["risk_polar_alt"] = np.clip(df["risk_score"] * df["alt_factor_polar"], 0, 1)
df["risk_mid_alt"]   = np.clip(df["risk_score"] * df["alt_factor_mid"],   0, 1)
df["risk_low_alt"]   = np.clip(df["risk_score"] * df["alt_factor_low"],   0, 1)

def classify(score):
    if score < 0.30:
        return "Low"
    elif score < 0.60:
        return "Moderate"
    else:
        return "High"

df["cat_polar_alt"] = df["risk_polar_alt"].apply(classify)
df["cat_mid_alt"]   = df["risk_mid_alt"].apply(classify)
df["cat_low_alt"]   = df["risk_low_alt"].apply(classify)

print("Altitude factors:",
      "polar", float(df["alt_factor_polar"].iloc[0]),
      "mid",   float(df["alt_factor_mid"].iloc[0]),
      "low",   float(df["alt_factor_low"].iloc[0]))

print("\nPolar (alt) category distribution:\n", df["cat_polar_alt"].value_counts())
print("\nMid (alt) category distribution:\n", df["cat_mid_alt"].value_counts())
print("\nLow (alt) category distribution:\n", df["cat_low_alt"].value_counts())


In [ ]:
#reroute possibilty 
import numpy as np

df = df.copy()

def logistic(x):
    return 1 / (1 + np.exp(-x))

def reroute_prob(risk, lat_class="polar"):
    """
    Simple reroute probability model.
    Tunable parameters:
      - polar has higher baseline sensitivity
      - mid/low are less sensitive
    """
    risk = np.asarray(risk, dtype=float)

    if lat_class == "polar":
        k, x0, bias = 12.0, 0.48, -0.6   # steeper, earlier action
    elif lat_class == "mid":
        k, x0, bias = 10.0, 0.55, -0.9
    else:  # low
        k, x0, bias = 9.0,  0.60, -1.1

    # logistic around threshold x0, with slight bias
    return logistic(k * (risk - x0) + bias)

df["p_reroute_polar"] = reroute_prob(df["risk_polar_alt"], "polar")
df["p_reroute_mid"]   = reroute_prob(df["risk_mid_alt"],   "mid")
df["p_reroute_low"]   = reroute_prob(df["risk_low_alt"],   "low")

# A simple operational decision rule:
# reroute if probability > 0.5 OR risk category High
df["reroute_flag_polar"] = (df["p_reroute_polar"] > 0.50) | (df["cat_polar_alt"] == "High")
df["reroute_flag_mid"]   = (df["p_reroute_mid"]   > 0.50) | (df["cat_mid_alt"]   == "High")
df["reroute_flag_low"]   = (df["p_reroute_low"]   > 0.50) | (df["cat_low_alt"]   == "High")

print("Reroute rates (% of hours flagged):")
print("Polar:", 100 * df["reroute_flag_polar"].mean())
print("Mid  :", 100 * df["reroute_flag_mid"].mean())
print("Low  :", 100 * df["reroute_flag_low"].mean())

print("\nExample rows (top reroute probability - polar):")
print(df.sort_values("p_reroute_polar", ascending=False)[
    ["timestamp_utc", "risk_score", "risk_polar_alt", "cat_polar_alt", "p_reroute_polar", "reroute_flag_polar"]
].head(10))


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")
OUT_DIR = ROOT / "models" / "phase2_operational_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PARQUET = ROOT / "data" / "processed" / "phase2_master_with_flight_ops_20100501_20191231.parquet"
df.to_parquet(OUT_PARQUET, index=False)
print("Saved:", OUT_PARQUET)


storm = df[
    (df["timestamp_utc"] >= "2017-09-05") &
    (df["timestamp_utc"] <= "2017-09-08")
].copy()

plt.figure(figsize=(12,5))
plt.plot(storm["timestamp_utc"], storm["p_reroute_polar"], label="Polar reroute probability")
plt.plot(storm["timestamp_utc"], storm["p_reroute_mid"],   label="Mid-lat reroute probability")
plt.legend()
plt.title("Reroute Probability During Sept 2017 Storm")
plt.xlabel("Time")
plt.ylabel("Probability (0–1)")
plt.tight_layout()
plt.savefig(OUT_DIR / "reroute_probability_storm_2017.png", dpi=300)
plt.close()


plt.figure(figsize=(12,4))
plt.plot(df["timestamp_utc"], df["risk_polar_alt"], label="Polar risk (alt-adjusted)")
plt.plot(df["timestamp_utc"], df["risk_mid_alt"],   label="Mid risk (alt-adjusted)")
plt.plot(df["timestamp_utc"], df["risk_low_alt"],   label="Low risk (alt-adjusted)")
plt.legend()
plt.title("Altitude-Adjusted Aviation Risk (2010–2019)")
plt.xlabel("Time")
plt.ylabel("Risk (0–1)")
plt.tight_layout()
plt.savefig(OUT_DIR / "altitude_adjusted_risk_2010_2019.png", dpi=300)
plt.close()

print("Plots saved to:", OUT_DIR)


In [ ]:
import sys
!{sys.executable} -m pip install plotly

In [ ]:
try:
    import plotly
    print(f"Success! Plotly version {plotly.__version__} is ready.")
except ImportError:
    print("Still not found.)

In [ ]:
fig.write_html(
    "heliogaina_phase2_3d_aviation_risk.html",
    include_plotlyjs="cdn"
)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")
FILE = ROOT / "data" / "processed" / "phase2_aviation_risk_20100501_20191231.parquet"

df = pd.read_parquet(FILE)

print("Rows:", len(df))
print("\nColumns:")
for c in df.columns:
    print(" -", c)

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


ROOT = Path(r"C:\Users\Owner\OneDrive\Desktop\solar_proj")
FILE = ROOT / "data" / "processed" / "phase2_aviation_risk_20100501_20191231.parquet"

df = pd.read_parquet(FILE)

df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values("timestamp_utc").reset_index(drop=True)

print("Loaded rows:", len(df))




storm = df[
    (df["timestamp_utc"] >= "2017-09-01") &
    (df["timestamp_utc"] <= "2017-09-10")
].copy()

storm_hourly = (
    storm.set_index("timestamp_utc")
         .resample("1H")
         .mean(numeric_only=True)
         .dropna()
)

plt.figure(figsize=(12,5))

plt.plot(storm_hourly.index, storm_hourly["risk_score"], 
         label="Final Aviation Risk Score", linewidth=2)

plt.plot(storm_hourly.index, storm_hourly["hf_blackout_risk"], 
         label="HF Blackout Proxy")

plt.plot(storm_hourly.index, storm_hourly["gps_disturbance_index"], 
         label="GPS Disturbance Index")

plt.plot(storm_hourly.index, storm_hourly["radiation_proxy"], 
         label="Radiation Proxy")

plt.title("Risk Decomposition During Sept 2017 Solar Storm")
plt.xlabel("Time (UTC)")
plt.ylabel("Normalized Score (0–1)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



driver_counts = df["dominant_driver"].value_counts()

plt.figure(figsize=(6,4))
plt.bar(driver_counts.index.astype(str), driver_counts.values)
plt.title("Dominant Risk Driver Frequency (2010–2019)")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nDominant Driver Percentages:")
print((driver_counts / len(df) * 100).round(2))



cat_counts = df["risk_category"].value_counts()

plt.figure(figsize=(6,4))
plt.bar(cat_counts.index.astype(str), cat_counts.values)
plt.title("Risk Category Distribution")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nRisk Category Percentages:")
print((cat_counts / len(df) * 100).round(2))



plt.figure(figsize=(12,4))
plt.plot(storm_hourly.index, storm_hourly["tec_anomaly"])
plt.title("TEC Anomaly During Sept 2017 Storm (GPS Impact Validation)")
plt.xlabel("Time (UTC)")
plt.ylabel("TEC Anomaly")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Risk Score Summary:")
print(df["risk_score"].describe())

print("\nHF Proxy Summary:")
print(df["hf_blackout_risk"].describe())

print("\nGPS Index Summary:")
print(df["gps_disturbance_index"].describe())